# Known-structure guided design — interactive results

Scaffolds that have an **experimental** RCSB structure (>=97% identity) are guided toward **~80%-identity O_train targets**,
using the real 5 A chromophore pocket (no ESMFold). Two cohorts:

- **S-train -> O-train** — scaffold in the surrogate *train* split (in-distribution)
- **S-test -> O-train** — scaffold in the surrogate *test* split (**generalization**)

Design: ESM-2 surrogate (`cnn-max-d1`) guides on `(ex_max, em_max)`, ProstT5 oracle (`cnn-max-d2`) judges; 3 iters, T=5, k=10, lambda=20.
This notebook loads the per-task trajectory CSVs and renders interactive plotly views (hover, zoom, per-task selector).

In [7]:
import os, sys, glob
sys.path.insert(0, os.path.abspath("."))      # parallel_pipeline (for common)
import numpy as np, pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import common as C                              # anchors cwd to peak_design/

pio_tmpl = "plotly_white"
COHORTS = {"knownstruct_Strain_Otrain": "S-train", "knownstruct_Sval_Otrain": "S-val",
           "knownstruct_Stest_Otrain": "S-test"}
COLOR = {"S-train": "#2b6c8f", "S-val": "#7a5ea8", "S-test": "#f6a200"}

# ---- load every per-task trajectory CSV -------------------------------------------
frames = []
for coh, label in COHORTS.items():
    for fn in sorted(glob.glob(os.path.join(C.PIPE_OUT, coh, "design_*.csv"))):
        df = pd.read_csv(fn); df["cohort"] = label
        frames.append(df)
traj = pd.concat(frames, ignore_index=True)
numcols = ["round", "peak_err", "pred_ex", "pred_em", "ppl", "ident_to_scaffold",
           "seq_id_scaf_target", "target_ex", "target_em", "scaffold_ex", "scaffold_em"]
traj[numcols] = traj[numcols].apply(pd.to_numeric, errors="coerce")

# ---- per-task summary: best round = min ORACLE peak error over rounds >= 1 ---------
def _best(g):
    scaf = g[g["round"] == 0].iloc[0]
    b = g[g["round"] >= 1].sort_values("peak_err").iloc[0]
    return pd.Series(dict(
        task=scaf["example"], cohort=scaf["cohort"], pdb=scaf["scaffold_pdb"],
        scaffold=scaf["scaffold_name"], target=scaf["target_name"], seq_id=scaf["seq_id_scaf_target"],
        scaf_err=scaf["peak_err"], best_round=int(b["round"]), best_err=b["peak_err"],
        d_err=b["peak_err"] - scaf["peak_err"], id_scaf=b["ident_to_scaffold"],
        ppl0=scaf["ppl"], ppl=b["ppl"],
        best_ex=b["pred_ex"], best_em=b["pred_em"], scaf_pred_ex=scaf["pred_ex"], scaf_pred_em=scaf["pred_em"],
        target_ex=scaf["target_ex"], target_em=scaf["target_em"]))
best = pd.DataFrame([_best(g) for _, g in traj.groupby("example", sort=False)]).reset_index(drop=True)
best["improved"] = best["d_err"] < 0

n = len(best)
print(f"{n} tasks | improved (oracle err down): {int(best['improved'].sum())}/{n}")
for label in COHORTS.values():
    s = best[best.cohort == label]
    if not len(s):
        continue
    print(f"  {label}: {int(s['improved'].sum())}/{len(s)} improved | "
          f"scaffold {s.scaf_err.mean():.1f} -> best {s.best_err.mean():.1f} nm (Delta {s.d_err.mean():+.1f}) | "
          f"id kept {s.id_scaf.mean():.0%}")
best.sort_values("d_err").head(3)[["task", "cohort", "seq_id", "scaf_err", "best_err", "d_err", "id_scaf"]]

60 tasks | improved (oracle err down): 45/60
  S-train: 13/20 improved | scaffold 25.6 -> best 20.0 nm (Delta -5.6) | id kept 91%
  S-val: 18/20 improved | scaffold 55.8 -> best 34.9 nm (Delta -20.9) | id kept 92%
  S-test: 14/20 improved | scaffold 31.8 -> best 15.4 nm (Delta -16.4) | id kept 92%


,task,cohort,seq_id,scaf_err,best_err,d_err,id_scaf
59,td-RFP639-mRubyFT,S-test,0.817,177.52,72.28,-105.24,0.900
23,RFP630-mRubyFT,S-val,0.821,170.79,85.74,-85.05,0.905
0,Azurite-AvicFP1,S-train,0.781,80.82,4.25,-76.57,0.920


## 1. Did design help? Scaffold vs best-design oracle error

Each point is a task; **below the dashed diagonal = improved** (best design closer to the target than the scaffold was).
Hover for scaffold→target, PDB, identities. Log axes handle the far-red outliers.

In [8]:
lim = float(max(best.scaf_err.max(), best.best_err.max())) * 1.15
fig = px.scatter(
    best, x="scaf_err", y="best_err", color="cohort", symbol="cohort",
    color_discrete_map=COLOR, hover_name="task", log_x=True, log_y=True,
    hover_data={"seq_id": ":.0%", "id_scaf": ":.0%", "pdb": True, "best_round": True,
                "scaf_err": ":.1f", "best_err": ":.1f", "cohort": False},
    labels={"scaf_err": "scaffold oracle err (nm)", "best_err": "best-design oracle err (nm)"},
    template=pio_tmpl, title="Best design vs scaffold — below diagonal = improved")
fig.add_shape(type="line", x0=0.3, y0=0.3, x1=lim, y1=lim, line=dict(dash="dash", color="#888"))
fig.update_traces(marker=dict(size=11, line=dict(width=0.6, color="black")))
fig.update_layout(width=760, height=560, legend_title_text="cohort")
fig.show()

## 2. Per-task error change (Δ oracle error)

Sorted; **negative (left, blue-ish) = design improved** the oracle peak error. The few large positive bars are the
far-red FPs whose chromophore pocket edits pushed the oracle prediction further off.

In [9]:
b2 = best.sort_values("d_err")
fig = px.bar(b2, x="task", y="d_err", color="cohort", color_discrete_map=COLOR,
             hover_data={"seq_id": ":.0%", "scaf_err": ":.1f", "best_err": ":.1f", "d_err": ":+.1f"},
             labels={"d_err": "Δ oracle err (nm)  (negative = better)", "task": ""},
             template=pio_tmpl, title="Per-task oracle-error change (best round vs scaffold)")
fig.add_hline(y=0, line_dash="dash", line_color="#c5474b")
fig.update_layout(width=980, height=520, xaxis_tickangle=-55, legend_title_text="cohort")
fig.show()

## 3. Per-task trajectory viewer (use the dropdown)

Oracle peak error vs design round for each task; **round 0 = scaffold**. Pick a task from the dropdown.
The scaffold-only baseline and the best round are annotated.

In [10]:
tasks = list(best.sort_values(["cohort", "d_err"])["task"])
fig = go.Figure()
for k, tk in enumerate(tasks):
    g = traj[traj.example == tk].sort_values("round")
    coh = g["cohort"].iloc[0]
    fig.add_trace(go.Scatter(
        x=g["round"], y=g["peak_err"], mode="lines+markers", name=tk, visible=(k == 0),
        line=dict(color=COLOR[coh], width=2.5), marker=dict(size=9),
        hovertemplate="round %{x}<br>oracle err %{y:.1f} nm<br>ppl %{customdata[0]:.1f}"
                      "<br>id->scaf %{customdata[1]:.0%}<extra></extra>",
        customdata=np.stack([g["ppl"], g["ident_to_scaffold"]], axis=-1)))

buttons = []
for k, tk in enumerate(tasks):
    vis = [i == k for i in range(len(tasks))]
    row = best[best.task == tk].iloc[0]
    ttl = (f"{tk}  [{row['cohort']}, {row['seq_id']:.0%} id, PDB {row['pdb']}]  |  "
           f"scaffold {row['scaf_err']:.1f} -> best {row['best_err']:.1f} nm (round {row['best_round']})")
    buttons.append(dict(label=tk, method="update", args=[{"visible": vis}, {"title": ttl}]))

first = best[best.task == tasks[0]].iloc[0]
fig.update_layout(
    updatemenus=[dict(buttons=buttons, x=1.02, y=1.0, xanchor="left", direction="down")],
    template=pio_tmpl, width=900, height=520,
    xaxis_title="design round (0 = scaffold)", yaxis_title="oracle peak error (nm)",
    title=(f"{tasks[0]}  [{first['cohort']}, {first['seq_id']:.0%} id, PDB {first['pdb']}]  |  "
           f"scaffold {first['scaf_err']:.1f} -> best {first['best_err']:.1f} nm (round {first['best_round']})"))
fig.show()

## 4. Movement in the (excitation, emission) plane

For every task: the **scaffold** oracle prediction (circle) is connected to the **best design** (diamond), which should
move toward the **target** (star). Short arrows landing on a star = success. Hover any point for details.

In [11]:
fig = go.Figure()
# scaffold -> best-design connector lines
for _, r in best.iterrows():
    fig.add_trace(go.Scatter(x=[r.scaf_pred_ex, r.best_ex], y=[r.scaf_pred_em, r.best_em],
                             mode="lines", line=dict(color="#cccccc", width=1),
                             showlegend=False, hoverinfo="skip"))
def _pts(dfx, ex, em, sym, name, size):
    fig.add_trace(go.Scatter(
        x=best[ex], y=best[em], mode="markers", name=name,
        marker=dict(symbol=sym, size=size, line=dict(width=0.6, color="black"),
                    color=[COLOR[c] for c in best["cohort"]]),
        text=best["task"], customdata=np.stack([best["seq_id"], best["d_err"]], axis=-1),
        hovertemplate="%{text}<br>"+name+" ex %{x:.0f} / em %{y:.0f} nm"
                      "<br>id %{customdata[0]:.0%}<br>Δerr %{customdata[1]:+.1f} nm<extra></extra>"))
_pts(best, "scaf_pred_ex", "scaf_pred_em", "circle", "scaffold (pred)", 9)
_pts(best, "best_ex", "best_em", "diamond", "best design (pred)", 11)
fig.add_trace(go.Scatter(x=best["target_ex"], y=best["target_em"], mode="markers", name="target (actual)",
                         marker=dict(symbol="star", size=15, color="#3b8a6e", line=dict(width=0.6, color="black")),
                         text=best["task"], hovertemplate="%{text}<br>target ex %{x:.0f} / em %{y:.0f} nm<extra></extra>"))
fig.update_layout(template=pio_tmpl, width=820, height=640,
                  xaxis_title="excitation max (nm)", yaxis_title="emission max (nm)",
                  title="Oracle (ex, em): scaffold → best design vs target")
fig.show()

## 5. Cost of the edits: sequence retained & naturalness

Left: how much identity to the scaffold each design keeps (edits are confined to the chromophore + 5 Å pocket).
Right: ESM-2 pseudo-perplexity of scaffold vs design (near/below the diagonal = design stayed as natural as the scaffold).

In [12]:
from plotly.subplots import make_subplots
fig = make_subplots(rows=1, cols=2, subplot_titles=("Identity retained to scaffold", "Naturalness (pseudo-perplexity)"))
for label in COHORTS.values():
    s = best[best.cohort == label]
    if not len(s):
        continue
    fig.add_trace(go.Histogram(x=s["id_scaf"], name=label, marker_color=COLOR[label], opacity=0.75, nbinsx=15), 1, 1)
    fig.add_trace(go.Scatter(x=s["ppl0"], y=s["ppl"], mode="markers", name=label, marker=dict(color=COLOR[label], size=10, line=dict(width=0.5, color="black")),
                             text=s["task"], hovertemplate="%{text}<br>scaffold ppl %{x:.1f} -> design %{y:.1f}<extra></extra>", showlegend=False), 1, 2)
pl = float(max(best.ppl0.max(), best.ppl.max())) * 1.05
fig.add_trace(go.Scatter(x=[1, pl], y=[1, pl], mode="lines", line=dict(dash="dash", color="#888"), showlegend=False), 1, 2)
fig.update_layout(barmode="overlay", template=pio_tmpl, width=980, height=440, legend_title_text="cohort")
fig.update_xaxes(title_text="identity to scaffold", tickformat=".0%", row=1, col=1)
fig.update_xaxes(title_text="scaffold ppl", row=1, col=2); fig.update_yaxes(title_text="design ppl", row=1, col=2)
fig.show()

# sortable summary table (also saved per-cohort by summarize_knownstruct.py)
best.sort_values("d_err")[["task", "cohort", "pdb", "seq_id", "scaf_err", "best_round",
                           "best_err", "d_err", "id_scaf", "ppl0", "ppl"]].round(3).reset_index(drop=True)

,task,cohort,pdb,seq_id,scaf_err,best_round,best_err,d_err,id_scaf,ppl0,ppl
0,td-RFP639-mRubyFT,S-test,3E5W,0.817,177.52,2,72.28,-105.24,0.900,17.58,16.66
1,RFP630-mRubyFT,S-val,1UIS,0.821,170.79,3,85.74,-85.05,0.905,17.76,17.27
2,Azurite-AvicFP1,S-train,5N9O,0.781,80.82,1,4.25,-76.57,0.920,16.74,16.28
3,sg42-AvicFP1,S-test,1GFL,0.787,80.63,3,5.53,-75.10,0.904,16.49,15.72
4,P4-AvicFP1,S-val,1W7S,0.798,74.38,2,3.17,-71.21,0.908,16.63,16.80
5,mRuby-Electra1,S-test,3U0L,0.814,151.07,1,80.53,-70.54,0.916,16.89,15.30
6,eqFP611-mRubyFT,S-train,1UIS,0.817,161.21,2,98.00,-63.21,0.909,17.80,17.24
7,TagBFP-Crimson,S-val,3M24,0.789,167.48,3,116.08,-51.40,0.918,16.84,15.96
8,RFP637-mRubyFT,S-val,3E5W,0.813,179.18,3,133.82,-45.36,0.896,17.44,17.14
9,lanRFP-ΔS83l-LanYFP,S-train,4JEO,0.741,43.27,1,4.71,-38.56,0.913,15.61,14.16


## 6. Distribution of the design errors

Histogram of the **oracle peak error of the best design** (nm) per cohort — the residual distance to the target after design. **Left:** design error split by cohort. **Right:** the scaffold (round-0) error vs the best-design error across all tasks — the leftward shift is the improvement design bought.

In [15]:
# ---- distribution of the design (best-round) oracle error, vs the scaffold baseline ----
print("oracle peak error (nm): scaffold -> best design")
for label in COHORTS.values():
    s = best[best.cohort == label]
    if not len(s):
        continue
    print(f"  {label:8}: n={len(s):2d}  scaffold mean {s.scaf_err.mean():5.1f} / median {s.scaf_err.median():5.1f}"
          f"   ->   design mean {s.best_err.mean():5.1f} / median {s.best_err.median():5.1f} nm")
print(f"  {'ALL':8}: n={len(best):2d}  scaffold mean {best.scaf_err.mean():5.1f} / median {best.scaf_err.median():5.1f}"
      f"   ->   design mean {best.best_err.mean():5.1f} / median {best.best_err.median():5.1f} nm")

from plotly.subplots import make_subplots
mx = float(best[["scaf_err", "best_err"]].max().max()) * 1.05
sz = mx / 22
fig = make_subplots(rows=1, cols=2, subplot_titles=("Design error by cohort", "Scaffold vs design error (all tasks)"))
# left: design error split by cohort
for label in COHORTS.values():
    s = best[best.cohort == label]
    if not len(s):
        continue
    fig.add_trace(go.Histogram(x=s["best_err"], xbins=dict(start=0, end=mx, size=sz),
                               name=label, marker_color=COLOR[label], opacity=0.75), 1, 1)
# right: scaffold (before) vs design (after) across all tasks
fig.add_trace(go.Histogram(x=best["scaf_err"], xbins=dict(start=0, end=mx, size=sz),
                           name="scaffold", marker_color="#bbbbbb", opacity=0.6), 1, 2)
fig.add_trace(go.Histogram(x=best["best_err"], xbins=dict(start=0, end=mx, size=sz),
                           name="design", marker_color="#2b6c8f", opacity=0.6), 1, 2)
fig.add_vline(x=float(best["best_err"].median()), line_dash="dash", line_color="#333", row=1, col=1,
              annotation_text=f"design median {best['best_err'].median():.0f} nm")
fig.update_xaxes(title_text="oracle peak error (nm)", row=1, col=1)
fig.update_xaxes(title_text="oracle peak error (nm)", row=1, col=2)
fig.update_yaxes(title_text="tasks", row=1, col=1)
fig.update_layout(barmode="overlay", template=pio_tmpl, width=1040, height=460, legend_title_text="")
fig.show()

oracle peak error (nm): scaffold -> best design
  S-train : n=20  scaffold mean  25.6 / median  13.4   ->   design mean  20.0 / median   4.5 nm
  S-val   : n=20  scaffold mean  55.8 / median  32.9   ->   design mean  34.9 / median  30.6 nm
  S-test  : n=20  scaffold mean  31.8 / median  12.2   ->   design mean  15.4 / median   7.6 nm
  ALL     : n=60  scaffold mean  37.7 / median  22.1   ->   design mean  23.4 / median   8.4 nm


## 7. Change in design error vs. how far the scaffold started from the target

**x** = the **scaffold** oracle error — how far the scaffold's predicted spectrum started from the target (the "difference between the original scaffold and the target"). **y** = the **change in error** from design, `Δ = best-design err − scaffold err` (negative = improved). The dashed `y = 0` line is no change; the dotted `y = −x` line is the best possible (design error driven to 0). Points hugging `y = −x` fully closed the gap; points near `y = 0` barely moved; points above `y = 0` regressed. A strong negative correlation means the farther the scaffold started from its target, the more design improved it.

In [16]:
rho = best["d_err"].corr(best["scaf_err"])
xmax = float(best.scaf_err.max()) * 1.08
fig = px.scatter(
    best, x="scaf_err", y="d_err", color="cohort", symbol="cohort",
    color_discrete_map=COLOR, hover_name="task",
    hover_data={"seq_id": ":.0%", "scaf_err": ":.1f", "best_err": ":.1f", "d_err": ":+.1f",
                "id_scaf": ":.0%", "cohort": False},
    labels={"scaf_err": "scaffold→target difference = scaffold oracle err (nm)",
            "d_err": "change in design error   Δ = best − scaffold (nm)"},
    template=pio_tmpl,
    title=f"Change in design error vs. scaffold→target difference  (Pearson r = {rho:.2f})")
fig.add_hline(y=0, line_dash="dash", line_color="#c5474b")                                   # no change
fig.add_shape(type="line", x0=0, y0=0, x1=xmax, y1=-xmax, line=dict(dash="dot", color="#888"))  # y=-x: design err -> 0
fig.add_annotation(x=xmax * 0.68, y=-xmax * 0.72, text="y = −x  (design err → 0)", showarrow=False,
                   font=dict(color="#888", size=11))
fig.update_traces(marker=dict(size=11, line=dict(width=0.6, color="black")))
fig.update_layout(width=820, height=600, legend_title_text="cohort", xaxis_range=[0, xmax])
fig.show()